<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_14_3_frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 14: Wrapping Up**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 14 Material

* Part 14.1: Model Drift [[Video]]() [[Notebook]](t81_558_class_14_1_drift.ipynb)
* Part 14.2: Dealing with Bias [[Video]]() [[Notebook]](t81_558_class_14_2_bias.ipynb)
* **Part 14.3: Other Deep Learning Frameworks** [[Video]]() [[Notebook]](t81_558_class_14_3_frameworks.ipynb)
* Part 14.4: Deploying a PyTorch Neural Network [[Video]]() [[Notebook]](t81_558_class_14_4_deploy.ipynb)
* Part 14.5: The Future of AI [[Video]]() [[Notebook]](t81_558_class_14_5_new_tech.ipynb)

# Google CoLab Instructions

The following code checks that Google CoLab is running and sets up the correct hardware settings for PyTorch.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: not using Google CoLab


Using device: mps


# Part 14.3: Other Deep Learning Frameworks

This course teaches deep learning with PyTorch, but PyTorch is one of several major frameworks, and a well-rounded practitioner should understand the landscape. Each framework embodies particular design choices, and each dominates in particular settings. Knowing what else exists helps you read code from other teams, choose the right tool for a job, and appreciate why PyTorch works the way it does.

The good news is that the core ideas transfer completely. Tensors, automatic differentiation, layers, loss functions, and optimizers appear in every framework; only the syntax and the execution model differ. Once you understand deep learning through one framework, learning another is mostly a matter of translating vocabulary. This section surveys the most important alternatives to PyTorch, illustrates the key conceptual divide between them, and shows the same small network expressed three ways.

## The Major Frameworks

**TensorFlow** was released by Google in 2015 and, for years, was the dominant framework in industry. Its original design built a static computation graph first and executed it afterward, which was efficient but awkward to debug. **Keras**, a high-level API now integrated with TensorFlow, made model building far more approachable and remains one of the most beginner-friendly ways to define networks. TensorFlow's mature deployment ecosystem, including TensorFlow Lite for mobile and TensorFlow Serving for production, keeps it widely used in industry.

**JAX**, also from Google, has become the framework of choice for a great deal of cutting-edge research. JAX pairs a NumPy-like interface with composable function transformations: `grad` for automatic differentiation, `jit` for just-in-time compilation to fast XLA code, and `vmap` for automatic vectorization. Higher-level libraries such as **Flax** and **Haiku** add neural-network layers on top. JAX's functional style suits large-scale experimentation and the training of very large models.

**PyTorch**, from Meta, is what this course uses and has become the most popular framework for research and, increasingly, production. Its define-by-run execution makes models feel like ordinary Python and easy to debug, while `torch.compile` and a rich ecosystem (PyTorch Lightning, Hugging Face, TorchServe) cover production needs. Other tools round out the picture: **ONNX** provides a common format for moving trained models between frameworks, and **Hugging Face Transformers** offers pretrained models that run across PyTorch, TensorFlow, and JAX.

## Eager Versus Graph Execution

The deepest conceptual difference among frameworks is *when* the computation runs. In **define-by-run** (eager) execution, which PyTorch popularized, each operation runs immediately as the Python interpreter reaches it. The network is just Python code, so you can print intermediate tensors, set breakpoints, and use ordinary control flow. This is what makes PyTorch so pleasant to debug.

In **define-and-run** (graph) execution, the original TensorFlow model, you first build a symbolic graph describing the whole computation and then hand it to an engine that runs and optimizes it. This is harder to inspect but can be faster and easier to deploy, because the optimizer sees the entire computation at once.

The two approaches have converged. TensorFlow 2 adopted eager execution by default, PyTorch added graph compilation through `torch.compile`, and JAX lets you write eager-style code that `jit` compiles into an optimized graph. Modern frameworks now try to give you the debuggability of eager execution and the performance of graphs. The small PyTorch model below runs eagerly, each line executing as it is reached.

In [2]:
import torch
import torch.nn as nn

# PyTorch: define-by-run. Each operation executes immediately.
model = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 3),
)

x = torch.randn(2, 4)
output = model(x)              # runs right now; we can inspect it immediately
print("output shape:", tuple(output.shape))
print("output:\n", output)

output shape: (2, 3)
output:
 tensor([[-0.4919, -0.3013,  0.1772],
        [-0.3335, -0.0254,  0.1537]], grad_fn=<AddmmBackward0>)


## The Same Model in Three Frameworks

To see how similar the frameworks really are, here is the same small classifier, a hidden layer with a ReLU followed by an output layer, written in PyTorch, Keras, and Flax (JAX). The code below is illustrative and is not executed here, since installing all three frameworks is unnecessary; the point is to notice how the same three ideas, a linear layer, an activation, and another linear layer, appear in each.

**PyTorch** (this course):

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)
```

**Keras** (TensorFlow):

```python
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(16, activation="relu", input_shape=(4,)),
    keras.layers.Dense(3),
])
```

**Flax** (JAX):

```python
import flax.linen as nn

class Model(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.relu(nn.Dense(16)(x))
        return nn.Dense(3)(x)
```

The three snippets are almost line-for-line translations. The layers have different names (`Linear`, `Dense`), and JAX's functional style looks a little different, but the architecture is identical. This is why the skills from this course transfer: you have learned deep learning, not merely PyTorch.

## Choosing a Framework

For most people the practical advice is simple: learn one framework well, and PyTorch is an excellent choice because of its readability, its enormous research community, and its strong production tooling. Reach for TensorFlow and Keras when you are working in an ecosystem built around them, especially for mobile or established enterprise deployment pipelines. Reach for JAX when you need its functional transforms and top-tier performance for large-scale research.

Whatever the framework, the concepts you carry between them are what matter. With the ecosystem in view, the next section returns to PyTorch specifically and addresses the final step of a project's life cycle: getting a trained model into production, in [Deploying a PyTorch Neural Network](t81_558_class_14_4_deploy.ipynb).